# ДЗ-1. Ваш участок ПРАЙМ

Вы — аналитик команды подписки ПРАЙМ. У каждого аналитика свой **участок**: одна таблица, один сегмент и одно окно времени. Это тот самый персональный срез из `prime.assignments`, с которым вы работали на первом семинаре, и больше он ни у кого не повторяется.

В понедельник руководитель просит сводку по участку — восемь задач. В каждой две части: **расчёт** (функция на Python) и **вопрос**, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N». Числа у всех разные, поэтому и выводы у каждого свои.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение к базе и выгрузка вашего участка | вписываете пять значений в ячейку 1.1 и запускаете ячейки по порядку |
| **2. Восемь задач** | условия, заготовки функций, вопросы | пишете код функций и ответы словами |
| **3. Итог** | готовый код: сводка ваших ответов и проверка их формата | запускаете перед сдачей |
| **4. Исследование** | две открытые задачи для тех, кто хочет 9 или 10 | по желанию |

**Оценка.** Части 1–3 — это оценка до 8 баллов: сделали всё аккуратно, выводы верные — 8, и это «отлично». 9 и 10 — только за исследование в части 4. Все критерии — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-1-окружение-и-репозиторий), там же — как сдавать.

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей — **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в **вашем** репозитории как `notebooks/hw1.ipynb`, а JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно — он готов. Единственная ваша правка — пять значений в ячейке 1.1. Номер ячейки — это номер заголовка над ней: «ячейка 1.1» — код сразу под заголовком «1.1 Ваш участок».

Код длинный, потому что аккуратно обходит несколько ловушек, до которых мы дойдём позже. Прочитать его полезно, разбирать каждую строку не обязательно.

| Ячейка | Что делает | Что делаете вы |
|---|---|---|
| 1.1 Ваш участок | хранит пять значений участка | вписываете значения |
| 1.2 Подключение | подключается к базе через ваш `.env` | запускаете |
| 1.3 Строка участка | показывает вашу строку из `prime.assignments` | запускаете, копируете значения в 1.1 и запускаете 1.1 ещё раз |
| 1.4 Запрос | описывает, как достать участок из любой из четырёх таблиц | запускаете |
| 1.5 Выгрузка | достаёт участок в переменную `rows` | запускаете — дальше вы работаете с `rows` |
| 1.6 Формат ответов | служебная функция для итога | запускаете |

### 1.1 Ваш участок

Впишите значения из своей строки `prime.assignments` — строками, в кавычках. Какие они, покажет ячейка 1.3: запустите 1.2 и 1.3, скопируйте напечатанные строки сюда и **запустите эту ячейку ещё раз**, иначе Python продолжит видеть старые значения.

Это **единственное место**, где указан участок. При проверке преподаватель подставит сюда другой участок — и все ответы должны пересчитаться сами. Поэтому ячейку не удаляйте и не пересоздавайте, а значения больше нигде не повторяйте.

In [ ]:
TABLE = "transactions" # таблица, например "payments"
SEGMENT_COLUMN = "channel"  # колонка сегмента, например "payment_method"
SEGMENT = "app"         # сегмент — значение в этой колонке, например "card"
PERIOD_START = "2025-08-01"    # первый день окна, например "2026-04-01"
PERIOD_END = "2025-08-31"      # последний день окна ВКЛЮЧИТЕЛЬНО, например "2026-06-30"

### 1.2 Подключение

Подключаемся к учебной базе и заводим функцию `query(sql)`: она отправляет SQL-запрос и возвращает результат списком словарей — одна строка результата, один словарь, как на семинарах. Пароля в ноутбуке нет: его знает только ваш `.env`.

Если здесь ошибка `Пакет prime не найден` — ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN` — нет файла `.env` в корне `prime-monitor`.

Соединение открывается на каждый запрос и сразу закрывается: так после сна ноутбука или смены Wi-Fi ничего не отваливается, а общий сервер не держит лишних подключений.

In [ ]:
import sys
from collections import Counter, defaultdict  # понадобятся в задачах
from datetime import date
from decimal import Decimal

from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            # Моменты времени в базе хранятся с часовым поясом. День события считаем
            # по UTC — так он у всех одинаковый, какие бы настройки ни стояли у вас.
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        # Соединение не держим: сервер у всей группы общий, а после сна ноутбука
        # старое соединение всё равно мёртвое.
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

### 1.3 Строка участка

Показывает вашу строку из `prime.assignments` — ту же, что на первом семинаре. Чужих строк база не показывает. Скопируйте напечатанное в ячейку 1.1 и запустите 1.1 ещё раз.

За то, что в 1.1 стоят именно ваши значения, отвечаете вы: код с чужим участком спокойно всё посчитает, только не то.

In [ ]:
mine = query("select * from prime.assignments where login = current_user")

if mine:
    row = mine[0]
    print("Ваш участок. Скопируйте в ячейку 1.1 и запустите её:\n")
    print(f"TABLE = {row['table_name']!r}")
    print(f"SEGMENT_COLUMN = {row['segment_column']!r}")
    print(f"SEGMENT = {row['segment']!r}")
    print(f"PERIOD_START = {str(row['period_start'])!r}")
    print(f"PERIOD_END = {str(row['period_end'])!r}")
else:
    # Так бывает, когда ноутбук запускает преподаватель под своей ролью.
    print("Строки участка для этого пользователя в базе нет.")

### 1.4 Запрос

Четыре таблицы устроены по-разному, но для задач нужно одно и то же. Поэтому запрос приводит любую из них к **пяти полям**:

| Поле | Что это |
|---|---|
| `row_id` | идентификатор строки |
| `actor_id` | участник — тот, чьи это строки |
| `day` | день события, тип `datetime.date` |
| `ok` | состоялось ли событие: `True` или `False` |
| `value` | сколько: деньги, минуты или часы, тип `Decimal` |

Что это значит в **вашей** таблице:

| Таблица | Одна строка — это | Участник | Сегмент | `ok = True` | `ok = False` | `value` |
|---|---|---|---|---|---|---|
| `payments` | попытка списать плату за подписку | подписка | способ оплаты: `card`, `sbp`, `wallet`, `balance` | списание прошло | не прошло или деньги вернули | сумма списания, ₽ |
| `transactions` | покупка клиента в партнёрском сервисе | клиент | канал: `app` — приложение, `web` — сайт, `pos` — касса | покупка прошла | отклонена или проведена и отменена | сумма покупки, ₽; бывает ноль и меньше нуля |
| `service_usage` | сессия в сервисе ПРАЙМ | клиент | устройство | сессия записана корректно: конец позже начала | конец не позже начала | длительность, минуты |
| `support_tickets` | обращение в поддержку | клиент | тема обращения | обращение решено | ещё в работе | время от создания до решения, часы; у нерешённых — `None` |

**Участник** здесь — просто тот, чьи это строки. Не путайте с участниками подписки из `subscription_members`, о которых шла речь на первом семинаре. А у `support_tickets` сумма `value` по дню растёт и от числа обращений, и от медленных решений — это не одно и то же.

Как работает запрос:

1. Блок `acts` выбирает **1000 участников** вашего сегмента за ваше окно. Это выборка, а не весь сегмент: поэтому строк в `rows` меньше, чем вы насчитали в хвосте первого семинара, — там был весь срез. Подгонять под то число не нужно. В ответах словами пишите «в выгрузке» или «среди 1000 подписок / клиентов», а не «по всем картам за квартал». Перемешивание через `md5` делает выбор похожим на случайный, но при каждом запуске — одинаковым.
2. Основной запрос берёт строки этих участников — **только в вашем сегменте и в вашем окне** (тот же фильтр, что в `acts`) — и переименовывает колонки в пять полей.
3. В `{фигурных скобках}` — названия колонок из словаря `COLUMNS`: они подставляются под вашу таблицу. После двоеточия (`:segment`, `:period_start`, `:period_end`) — ваши значения из ячейки 1.1.
4. Граница окна: момент события `>=` первого дня и `<` дня, следующего за последним. Так последний день попадает в окно целиком.

In [ ]:
# Как в каждой из четырёх таблиц называются колонки, из которых получаются пять полей.
COLUMNS = {
    "payments": {
        "segment_column": "payment_method",   # колонка сегмента
        "row_id": "payment_id",                # -> row_id
        "actor": "subscription_id",            # -> actor_id
        "moment": "paid_at",                   # момент события -> day
        "ok": "status = 'success'",            # -> ok
        "value": "amount",                     # -> value
    },
    "transactions": {
        "segment_column": "channel",
        "row_id": "transaction_id",
        "actor": "client_id",
        "moment": "occurred_at",
        "ok": "status = 'success'",
        "value": "amount",
    },
    "service_usage": {
        "segment_column": "device_type",
        "row_id": "usage_id",
        "actor": "client_id",
        "moment": "started_at",
        "ok": "ended_at > started_at",
        "value": "extract(epoch from (ended_at - started_at)) / 60.0",      # секунды -> минуты
    },
    "support_tickets": {
        "segment_column": "category",
        "row_id": "ticket_id",
        "actor": "client_id",
        "moment": "created_at",
        "ok": "resolved_at is not null",
        "value": "extract(epoch from (resolved_at - created_at)) / 3600.0",  # секунды -> часы
    },
}

SLICE_SQL = """
with acts as (
    select actor_id
    from (
        select distinct {actor}::text as actor_id
        from prime.{table}
        where {segment_column} = :segment
          and {moment} >= cast(:period_start as date)
          and {moment} <  cast(:period_end as date) + 1
    ) s
    order by md5(actor_id)
    limit 1000
)
select t.{row_id}::text as row_id,
       t.{actor}::text  as actor_id,
       t.{moment}::date as day,
       {ok}             as ok,
       {value}          as value
from prime.{table} t
join acts a on a.actor_id = t.{actor}::text
where t.{segment_column} = :segment
  and t.{moment} >= cast(:period_start as date)
  and t.{moment} <  cast(:period_end as date) + 1
order by t.{row_id}
"""

print("запрос описан")

### 1.5 Выгрузка

Достаём участок из базы в переменную `rows` — список словарей, по одному на строку. **Все задачи ниже работают с `rows`.**

Сколько в выгрузке строк, участников и дней — ячейка не печатает: это вы посчитаете сами в задаче 1.

In [ ]:
# В текст запроса подставляются только известные названия таблиц и колонок —
# поэтому сначала проверяем, что значения в ячейке 1.1 из этого списка.
if TABLE not in COLUMNS:
    raise ValueError(f"TABLE должна быть одной из {list(COLUMNS)}, сейчас {TABLE!r}")
if SEGMENT_COLUMN != COLUMNS[TABLE]["segment_column"]:
    raise ValueError(f"у таблицы {TABLE} колонка сегмента называется {COLUMNS[TABLE]['segment_column']!r}")
try:
    window_days = (date.fromisoformat(PERIOD_END) - date.fromisoformat(PERIOD_START)).days + 1
except (TypeError, ValueError):
    raise ValueError("PERIOD_START и PERIOD_END — строки вида '2025-04-01'") from None

# Собираем запрос под вашу таблицу и выполняем. Обычно несколько секунд;
# если висит дольше минуты — см. «Если что-то не работает» в тексте задания.
slice_sql = SLICE_SQL.format(table=TABLE, **COLUMNS[TABLE])
rows = query(slice_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)

# Защита от ошибок в значениях 1.1: пустая выгрузка или дни за пределами окна.
if not rows:
    raise ValueError("выгрузка пустая — проверьте значения в ячейке 1.1")
if len({row["day"] for row in rows}) > window_days:
    raise ValueError("в выгрузке дней больше, чем в окне — граница периода съехала")

print("Выгрузка готова. Так выглядят первые три строки:")
rows[:3]

### 1.6 Формат ответов

Служебная функция: по ней итог в части 3 проверяет, что ответ записан в нужном формате — список, число, дата строкой. **Верно ли посчитано, она не знает и не сообщает** — это проверяет преподаватель. Код ячейки свёрнут — так задумано, просто запустите её.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "list из 4 элементов: [int, int, float до 2 знаков, int]",
    2: "int",
    3: "dict: ключи из Mon…Sun, значения float до 2 знаков",
    4: "list [str, int]",
    5: "list из 3 элементов, каждый — list ['ГГГГ-ММ-ДД', float до 2 знаков]",
    6: "list [int, 'ГГГГ-ММ-ДД']",
    7: "list [float до 2 знаков, int]",
    8: "float до 2 знаков",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:  # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def iso_day(x) -> bool:  # дата строкой 'ГГГГ-ММ-ДД'
        try:
            return type(x) is str and date.fromisoformat(x).isoformat() == x
        except ValueError:
            return False

    if answer is None:
        return "не решена: функция вернула None — в ней остался ... или забыт return"
    checks = {
        1: lambda a: type(a) is list and len(a) == 4 and type(a[0]) is int and type(a[1]) is int
                     and money(a[2]) and type(a[3]) is int,
        2: lambda a: type(a) is int,
        3: lambda a: type(a) is dict and set(a) <= {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"}
                     and all(money(v) for v in a.values()),
        4: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is str and type(a[1]) is int,
        5: lambda a: type(a) is list and len(a) == 3
                     and all(type(p) is list and len(p) == 2 and iso_day(p[0]) and money(p[1]) for p in a),
        6: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is int and iso_day(a[1]),
        7: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and type(a[1]) is int,
        8: lambda a: money(a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r}"


print("проверка формата готова")

## Часть 2. Восемь задач

У каждой задачи два шага:

1. **Посчитать.** Допишите функцию `taskN(rows)` — она должна вернуть ответ ровно в указанном формате. Имена функций и переменных `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом участке.
2. **Ответить на вопрос.** Для ответа почти всегда нужно досчитать одно-два числа — для этого под вопросом есть пустая ячейка. Сам ответ пишется в ячейке «✍️ Ответ на вопрос N»: дважды щёлкните по ней, **заголовок не трогайте**, замените строку курсивом своим текстом и нажмите `Shift + Enter`.

**Ячейки расчётов тоже выполняются на контрольном участке.** Считайте в них от `rows` и `answer_N`; конкретные даты, `actor_id` и сегмент руками не вписывайте — на другом участке их нет, и ячейка упадёт.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All, и ноутбук не проходит целиком — это минус баллы за воспроизводимость. Верните в тело функции `...`: она вернёт `None`, итог покажет «❌ не решена», а остальное проверится как обычно.

**Про ответы словами.** Код можно писать вместе с ИИ-ассистентом — это нормально, отметьте это в декларации. А ответы пишите сами: здесь оценивается, как думаете вы. **Текст, написанный LLM, получает 0 баллов.** Хороший ответ — 2–4 предложения, в них ваши числа и ваш вывод. Сомнительный вывод баллов не приносит, даже если числа верные.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 3 проверяет только формат. Правильность проверяет преподаватель — на вашем участке и на контрольном.

### Три особенности данных, на которых спотыкаются чаще всего

* **`value` приходит типом `Decimal`, а не `float`.** Это точный тип для денег. `Decimal` складывается с `Decimal` и с целыми числами, а с `float` — нет: `Decimal("1.5") + 0.5` падает с `TypeError`. В ноутбуке С2-идиомы суммы копились от `total = 0.0` — здесь так не выйдет. Копите суммы в `Decimal` (`defaultdict(Decimal)`, `sum(...)` без стартового `0.0`), а к `float` приводите в самом конце: `float(round(сумма, 2))`.
* **В `support_tickets` у нерешённых обращений `value` равно `None`** — времени до решения ещё нет. Проверяйте `row["value"] is not None` там, где складываете `value`, а не `if row["value"]`: второе молча выкинет и честный ноль. Сами строки с `None` из `rows` не удаляйте — это нерешённые обращения, и они нужны в задачах 1, 2 и в вопросе 1.
* **Строки с `ok = True` и отрицательным или нулевым `value` не выбрасывайте.** В `transactions` такие есть — например, покупка на ноль рублей. Считайте их как есть.

Нужные приёмы разобраны в [ноутбуке С2-идиомы](https://github.com/tikhomirovd/python-for-ba-hse-2026/blob/master/02-среда-git-python/семинар/С2-идиомы.ipynb): генераторные выражения, множества, `defaultdict` и `Counter`, `sorted` с ключом, `enumerate`, `try/except`.

### Задача 1. Паспорт участка

Руководитель начинает каждую неделю с одного вопроса: «Сколько у нас всего — и сколько из этого настоящего?» Прежде чем что-то анализировать, надо знать масштаб участка.

**Что посчитать**

Четыре числа по вашей выгрузке `rows`:

1. сколько всего строк;
2. сколько строк с `ok = True` — событий, которые состоялись;
3. сколько `value` набежало по строкам с `ok = True` — выручка, минуты или часы, смотря какая у вас таблица;
4. сколько **различных** дней встречается в выгрузке — по всем строкам, а не только по состоявшимся.

**Формат ответа:** `[всего, с_ok, сумма, дней]` — например `[2222, 2100, 123456.7, 17]`: целое, целое, `float` до двух знаков, целое.

**Подсказка:** `len(rows)`; `sum(1 for row in rows if ...)`; множество дней `{row["day"] for row in rows}`.

In [ ]:
def task1(rows: list[dict]) -> list:
    # верните [всего, с_ok, сумма, дней]
    total = len(rows)
    ok_count = sum(1 for row in rows if row["ok"])
    total_value = sum(
        row["value"]
        for row in rows
        if row["ok"] and row["value"] is not None
  )
    days_count = len({row["day"] for row in rows})
    return [total, ok_count, float(round(total_value, 2)), days_count]

answer_1 = task1(rows)
answer_1

**Вопрос 1.** **Что не случилось.** Какая доля строк вашей выгрузки не состоялась (`ok = False`)? Что это значит в жизни вашей таблицы — что именно не произошло? Сколько `value` приходится на эти строки — или объясните, почему у вашей таблицы это число не имеет смысла или его нельзя посчитать.

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — по rows и answer_1.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
not_ok_count = answer_1[0] - answer_1[1]
not_ok_percent = not_ok_count / answer_1[0] * 100

not_ok_value = sum(
    row["value"]
    for row in rows
    if not row["ok"] and row["value"] is not None
)

print("Несостоявшихся покупок:", not_ok_count)
print("Доля, %:", round(not_ok_percent, 2))
print("Сумма по этим строкам, ₽:", float(round(not_ok_value, 2)))

#### ✍️ Ответ на вопрос 1

Доля строк, у которых ok = False по выборке покупок через приложение за август 2025 г. считается так: (все строки (total) - строки, у которых ok = True (ok_count)) / все строки. Если подставить значения: (1 557 - 1 515) / 1557 ≈ 0,02697 (или 2,7%). Эта доля означает, что 2,7% покупок в приложении отклонены или проведены и отменены.
Сумма по этим строкам (value) = 52 626,03 руб., считалось как сумма покупок по моей выборке с условием, что не ok = True и без None.

### Задача 2. И получилось, и нет

Самые интересные участники — те, у кого в одном окне было и «получилось», и «не получилось»: оплата сначала не прошла, а потом прошла; одну покупку отклонили, другую провели. Руководитель хочет знать, сколько таких.

**Что посчитать**

Сколько **различных** участников (`actor_id`) имеют в выгрузке хотя бы одну строку с `ok = True` **и** хотя бы одну строку с `ok = False`.

**Формат ответа:** целое число, например `123`. Ноль — тоже законный ответ.

**Подсказка:** два множества участников и операция `&` между ними.

In [ ]:
def task2(rows: list[dict]) -> int:
    # верните число участников
    ok_actors = {row["actor_id"] for row in rows if row["ok"]}
    not_ok_actors = {row["actor_id"] for row in rows if not row["ok"]}
    both_actors = ok_actors & not_ok_actors
    return len(both_actors)
answer_2 = task2(rows)
answer_2

**Вопрос 2.** **Почему столько.** Сколько в среднем строк приходится на одного участника в вашей выгрузке? Объясните, почему ответ задачи 2 получился именно таким. Если это ноль или единица — это не ошибка: объясните, откуда он берётся. Подумайте, к чему относится сегмент: к отдельной строке (у одного участника строки могут быть с разными значениями) или к участнику целиком (у всех его строк оно одно).

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — по rows и answer_2.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
actors_count = len({row["actor_id"] for row in rows})
average_rows = len(rows) / actors_count
print("Всего строк:", len(rows))
print("Уникальных клиентов:", actors_count)
print("В среднем строк на клиента:", round(average_rows, 3))
print("Клиентов с обоими результатами:", answer_2)

#### ✍️ Ответ на вопрос 2
В среднем 1,557 строк приходится на одного участника в моей выгрузке. Получилось 26 клиентов, потому что по ним проводились операции как успешные (то есть у которых ок = True), так и не успешные (ок = False), получили как пересечение двух множеств идентификаторов клиентов. Сегмент относится к отдельной строке, потому что клиент может покупать и через другие каналы, не только через приложение, которое в моей выборке.

### Задача 3. Главный день недели

Команда решает, в какой день недели ставить дежурство и запускать рассылки. Нужна раскладка участка по дням недели.

**Что посчитать**

Сумма `value` по строкам с `ok = True` — отдельно для каждого дня недели.

**Формат ответа:** словарь `{'Mon': 12345.67, 'Tue': ...}` — только дни недели, которые встретились; суммы `float` до двух знаков. Название дня берите из списка `WEEKDAYS` по номеру `row["day"].weekday()` (понедельник — 0).

**Подсказка:** `defaultdict(Decimal)`. Почему не `day.strftime('%a')`: результат зависит от языковых настроек — стоит где-нибудь вызвать `locale.setlocale`, и вместо `'Tue'` получится `'вт'`, а ответ разойдётся с форматом.

In [ ]:
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]  # WEEKDAYS[0] — понедельник


def task3(rows: list[dict]) -> dict:
    # верните {'Mon': сумма, 'Tue': сумма, ...}
    by_weekday = defaultdict(Decimal)
    for row in rows:
        if row["ok"] and row["value"] is not None:
            day = WEEKDAYS[row["day"].weekday()]
            by_weekday[day] += row["value"]
    return {
        day: float(round(total, 2))
        for day, total in by_weekday.items()
    }
answer_3 = task3(rows)
answer_3

**Вопрос 3.** **Сумма или среднее.** Сколько **различных дат** вашей выгрузки приходится на каждый день недели? Считайте даты, а не строки. Какой день лидирует по сумме из задачи 3, а какой — по средней сумме на одну такую дату (сумма из задачи 3, делённая на число дат этого дня недели)? Если лидеры разные — какой ответ вы понесли бы руководителю и почему; если одинаковые — объясните, почему так вышло.

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — по rows и answer_3.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
unique_dates = {row["day"] for row in rows}
dates_count = Counter(WEEKDAYS[day.weekday()] for day in unique_dates)

average_by_weekday = {}

for day in WEEKDAYS:
    if day in dates_count:
        total = answer_3.get(day, 0.0)
        average = total / dates_count[day]
        average_by_weekday[day] = average

        print(
            day,
            "| дат:", dates_count[day],
            "| сумма:", total,
            "| среднее на дату:", round(average, 2)
        )
max_total = max(answer_3.values())
sum_leaders = [
    day for day, total in answer_3.items()
    if total == max_total
]
max_average = max(average_by_weekday.values())
average_leaders = [
    day for day, average in average_by_weekday.items()
    if average == max_average
]
print("Лидеры по сумме:", sum_leaders)
print("Лидеры по среднему:", average_leaders)

#### ✍️ Ответ на вопрос 3
На понедельник, вторник, среду и четверг приходится 4 даты, на пятницу, субботу и воскресенье 5 дат (различия в количестве дат из-за того, что месяц начинается с пятницы). По итогам задачи 3 лидирует воскресенье - 342 439,47 руб., по средней сумме лидирует понедельник - 78 238,61 руб. Руководителю я бы понес среднюю сумму на дату, потому что сравнение по суммам не учитывает разницу в количестве дат, поэтому по результатам задачи 3 наибольшем по сумме покупок оказалось воскресенье, но по среднему больше оказался понедельник.

### Задача 4. Самый активный участник

Маркетинг хочет наградить самого активного участника участка — того, у кого больше всего состоявшихся событий. Приз уже куплен.

**Что посчитать**

Участник с наибольшим числом строк с `ok = True` и это число. Если таких несколько — тот, чей `actor_id` меньше при сравнении как текст (так сравнивает `min`).

**Формат ответа:** `[actor_id, число строк]` — например `['aaaaaaaa-1111-2222-3333-bbbbbbbbbbbb', 9]`: текст (`str`) и целое.

**Подсказка:** `Counter` по строкам с `ok = True` считает, сколько состоявшихся событий у каждого участника; `max` находит наибольшее число, `min` — наименьший `actor_id` среди тех, у кого оно такое.

In [ ]:
def task4(rows: list[dict]) -> list:
    # верните [actor_id, число строк]
    counts = Counter(
        row["actor_id"]
        for row in rows
        if row["ok"]
    )
    max_count = max(counts.values())
    winner = min(
        actor_id
        for actor_id, count in counts.items()
        if count == max_count
    )
    return [winner, max_count]
answer_4 = task4(rows)
answer_4

**Вопрос 4.** **Кто победил на самом деле.** Сколько участников делят первое место? Если больше одного — что на самом деле выбрало вашего «победителя»: данные или правило? И что в вашей таблице вообще значит «самый активный» — стоит ли маркетингу вручать приз по такому рейтингу?

In [ ]:
counts = Counter(
    row["actor_id"]
    for row in rows
    if row["ok"]
)
max_count = answer_4[1]
leaders_count = sum(
    1 for count in counts.values()
    if count == max_count
)
print("Максимальное число состоявшихся покупок:", max_count)
print("Столько клиентов делят первое место:", leaders_count)
print("Выбранный клиент:", answer_4[0])

In [ ]:
import pandas as pd

counts = Counter()
amounts = defaultdict(Decimal)

for row in rows:
    if row["ok"]:
        actor_id = row["actor_id"]
        counts[actor_id] += 1

        if row["value"] is not None:
            amounts[actor_id] += row["value"]

top_ten = sorted(
    counts.items(),
    key=lambda item: (-item[1], item[0])
)[:10]

print("Топ-10 клиентов по состоявшимся покупкам:")

for place, (actor_id, count) in enumerate(top_ten, start=1):
    total = float(round(amounts[actor_id], 2))

    print(
        f"{place}. Клиент: {actor_id}"
        f" | покупок: {count}"
        f" | сумма: {total:.2f} ₽"
    )

all_actors = {row["actor_id"] for row in rows}

purchase_counts = pd.Series(
    [counts.get(actor_id, 0) for actor_id in all_actors],
    name="Состоявшиеся покупки"
)

print("\nСтатистика по всем клиентам выгрузки:")
print(purchase_counts.describe().round(3))

In [ ]:
amounts = defaultdict(Decimal)
counts = Counter()

for row in rows:
    if row["ok"]:
        actor_id = row["actor_id"]
        counts[actor_id] += 1

        if row["value"] is not None:
            amounts[actor_id] += row["value"]

max_amount = max(amounts.values())

leaders = sorted(
    actor_id
    for actor_id, total in amounts.items()
    if total == max_amount
)

for actor_id in leaders:
    print("Клиент:", actor_id)
    print("Сумма состоявшихся покупок, ₽:", float(round(max_amount, 2)))
    print("Количество состоявшихся покупок:", counts[actor_id])

print("Клиентов с максимальной суммой:", len(leaders))

#### ✍️ Ответ на вопрос 4

В моей выборке только один победитель. В моем случае самый активный участник - клиент, который совершил наибольшее количество успешных покупок в приложении за август 2025 года - 11. Для определения нужен ли приз нужны дополнительные данные, поэтому я посмотрел топ 10 по количеству покупок и суммировал величину покупок. Оказалось, что топ 1 имеет небольшой отрыв от остальных клиентов. Также посмотрел первое место по сумме покупок, у клиента было всего 4 покупки, но сумма (31 038,29) в 2,36 раза больше, чем у первого места по количеству. Итоговое решение давать приз или нет - надо решать из целей компании, смотреть какое поведение приносит больше денег - частота покупок или величина покупок (что важнее, то и надо поощрять). 

### Задача 5. Три пиковых дня

Чтобы планировать нагрузку на команду и серверы, нужно знать пиковые дни окна.

**Что посчитать**

Три даты с наибольшей суммой `value` по строкам с `ok = True` — по убыванию суммы. Если суммы равны, раньше идёт более ранняя дата.

**Формат ответа:** `[['2024-03-15', 98765.43], ['2024-03-02', 91200.1], ['2024-03-21', 90000.0]]` — дата строкой `'ГГГГ-ММ-ДД'`, сумма `float` до двух знаков.

**Подсказка:** сначала словарь «день → сумма», потом `sorted(..., key=...)`: ключ-кортеж `(-сумма, день)` сортирует по убыванию суммы, а при равенстве — по возрастанию даты. Дата строкой — `day.isoformat()`.

In [ ]:
def task5(rows: list[dict]) -> list:
    # верните [['ГГГГ-ММ-ДД', сумма], [...], [...]]
    by_day = defaultdict(Decimal)
    for row in rows:
        if row["ok"] and row["value"] is not None:
            by_day[row["day"]] += row["value"]
    top_days = sorted(
        by_day.items(),
        key=lambda item: (-item[1], item[0])
    )[:3]
    return [
        [day.isoformat(), float(round(total, 2))]
        for day, total in top_days
    ]
answer_5 = task5(rows)
answer_5

**Вопрос 5.** **Откуда пик.** Какие это дни недели, сколько среди них суббот и воскресений? Сформулируйте одну гипотезу, почему пик пришёлся на эти даты, и напишите, какой расчёт на данных её подтвердил бы или опроверг.

In [ ]:
weekend_count = 0

for day_text, total in answer_5:
    day = date.fromisoformat(day_text)
    weekday_number = day.weekday()
    weekday_name = WEEKDAYS[weekday_number]

    if weekday_number >= 5:
        weekend_count += 1

    print(
        "Дата:", day_text,
        "| день недели:", weekday_name,
        "| сумма:", total
    )

print("Суббот и воскресений среди пиковых дат:", weekend_count)

In [ ]:
counts_by_day = Counter()
amounts_by_day = defaultdict(Decimal)

for row in rows:
    if row["ok"] and row["value"] is not None:
        day = row["day"]
        counts_by_day[day] += 1
        amounts_by_day[day] += row["value"]

peak_days = {
    date.fromisoformat(day_text)
    for day_text, total in answer_5
}

all_days = {row["day"] for row in rows}
other_days = all_days - peak_days

other_count = sum(counts_by_day[day] for day in other_days)
other_amount = sum(amounts_by_day[day] for day in other_days)

average_count = other_count / len(other_days) if other_days else 0
average_purchase = other_amount / other_count if other_count else 0

print("Остальные даты:")
print("Количество дат:", len(other_days))
print("В среднем покупок за дату:", round(average_count, 2))
print("Средняя сумма одной покупки, ₽:", float(round(average_purchase, 2)))

print("\nПиковые даты:")

for day_text, total in answer_5:
    day = date.fromisoformat(day_text)
    count = counts_by_day[day]
    amount = amounts_by_day[day]
    average = amount / count if count else 0

    print(
        "Дата:", day_text,
        "| покупок:", count,
        "| общая сумма, ₽:", float(round(amount, 2)),
        "| средняя сумма покупки, ₽:", float(round(average, 2))
    )

In [ ]:
from statistics import median

counts_by_day = Counter()
values_by_day = defaultdict(list)

for row in rows:
    if row["ok"]:
        day = row["day"]
        counts_by_day[day] += 1

        if row["value"] is not None:
            values_by_day[day].append(row["value"])

peak_days = {
    date.fromisoformat(day_text)
    for day_text, total in answer_5
}

all_days = {row["day"] for row in rows}
other_days = all_days - peak_days

other_count = sum(counts_by_day[day] for day in other_days)

other_values = []
for day in other_days:
    other_values.extend(values_by_day[day])

other_sum = sum(other_values)

print("ОСТАЛЬНЫЕ ДАТЫ")
print("Количество дат:", len(other_days))
print("Всего состоявшихся покупок:", other_count)
print("Общая сумма, ₽:", float(round(other_sum, 2)))

if other_days:
    average_count = other_count / len(other_days)
    print("В среднем покупок за дату:", round(average_count, 2))
    print(
        "В среднем сумма за дату, ₽:",
        float(round(other_sum / len(other_days), 2))
    )

if other_values:
    print(
        "Средняя сумма покупки, ₽:",
        float(round(other_sum / len(other_values), 2))
    )
    print(
        "Медиана суммы покупки, ₽:",
        float(round(median(other_values), 2))
    )

print("\nПИКОВЫЕ ДАТЫ")

for day_text, total in answer_5:
    day = date.fromisoformat(day_text)
    count = counts_by_day[day]
    values = values_by_day[day]
    day_sum = sum(values)

    print("\nДата:", day_text)
    print("День недели:", WEEKDAYS[day.weekday()])
    print("Состоявшихся покупок:", count)
    print("Общая сумма, ₽:", float(round(day_sum, 2)))

    if values:
        print(
            "Средняя сумма покупки, ₽:",
            float(round(day_sum / len(values), 2))
        )
        print(
            "Медиана суммы покупки, ₽:",
            float(round(median(values), 2))
        )

    if other_days:
        print("Количество выше среднего:", count > average_count)

#### ✍️ Ответ на вопрос 5

Из пиков - понедельник, вторник и один выходной - суббота. Гипотеза такая - три пика связаны с большим количеством покупок в эти дни, чем в среднем. Проверить гипотезу можно посчитав количество покупок, суммой покупок, средней суммой покупок и медианной суммы покупок пиков и оставшихся 28 дней. Проведя расчет, можно заметить, что количество покупок в пики сопоставимо с остальными днями (62 и 50 выше 48,57, 43 ниже 48,57) - гипотеза не подтверждена. Также можно заметить, что средняя сумма одной покупки у пиков выше, но медианное относительно остальных дней ненамного больше (кроме субботы). Скорее всего это влияние крупных покупок в пики.

### Задача 6. Когда набралась половина

Финансам важно, равномерно ли набирается сумма внутри окна: к какому дню набегает половина суммы по состоявшимся событиям.

**Что посчитать**

1. Возьмите дни, в которых есть хотя бы одна строка с `ok = True`, и расставьте по возрастанию даты.
2. Идите по ним и копите сумму `value` по строкам с `ok = True`.
3. Найдите первый день, на котором накопленное **достигло половины** суммы из пункта 3 задачи 1 (всё `value` по строкам с `ok = True`) или превысило её, и его порядковый номер в списке дней из шага 1, считая с 1.

**Формат ответа:** `[номер, 'ГГГГ-ММ-ДД']` — например `[12, '2024-03-15']`.

**Подсказка:** `sums` — словарь «день → сумма», как в задаче 5. `for number, day in enumerate(sorted(sums), start=1):` — `enumerate` сам считает номер.

In [ ]:
def task6(rows: list[dict]) -> list:
    # верните [номер дня, 'ГГГГ-ММ-ДД']
    by_day = defaultdict(Decimal)
    for row in rows:
        if row["ok"]:
            by_day[row["day"]] += (
                row["value"] if row["value"] is not None else 0
            )
    half_total = sum(by_day.values()) / 2
    accumulated = Decimal(0)

    for number, day in enumerate(sorted(by_day), start=1):
        accumulated += by_day[day]

        if accumulated >= half_total:
            return [number, day.isoformat()]
answer_6 = task6(rows)
answer_6

**Вопрос 6.** **Равномерно ли.** Какую долю дней заняла первая половина суммы: номер дня из задачи 6, делённый на число дней в списке из шага 1? Номер дня целый, поэтому при равномерном накоплении получилось бы около половины, округлённой вверх (15 из 30, 16 из 31) — сравнивайте с этим, а не с ровными 50 %. У вас раньше или позже — и что это говорит о том, как менялся участок внутри окна? Если вышло у середины — так и напишите.

In [ ]:
# Числа для ответа на вопрос 6 считайте здесь — по rows и answer_6.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
ok_days = {row["day"] for row in rows if row["ok"]}
days_count = len(ok_days)
half_number = answer_6[0]
half_date = answer_6[1]
days_percent = half_number / days_count * 100
middle_number = (days_count + 1) // 2
print("Дней с состоявшимися покупками:", days_count)
print("Половина суммы достигнута:", half_date)
print("Порядковый номер этого дня:", half_number)
print("Доля дней, %:", round(days_percent, 2))
print("Середина списка дней с округлением вверх:", middle_number)
print("Отклонение от середины:", half_number - middle_number)

#### ✍️ Ответ на вопрос 6

Долю дней заняла первая половина суммы (16 дней из 31) - 51,61%. Получилось у середины.

### Задача 7. Функция, которая не падает

Соседний отдел присылает суммы строками — так, как их набрал оператор: с запятой, с пробелами, иногда с прочерком. Нужна функция, которая превращает такую строку в число и не роняет ежедневный отчёт. Данные в этой задаче у всех одинаковые — список `RAW`.

**Что посчитать**

1. Функция `to_value(x)`: получает строку (или `None`) и возвращает `float` — или `None`, если число разобрать нельзя. Падать нельзя ни на одном значении. Правила записи:
   * точка и запятая — десятичный разделитель: `'89.10'` → `89.1`, `'349,90'` → `349.9`;
   * пробел и неразрывный пробел внутри числа — разделители тысяч: `'1 499,50'` → `1499.5`;
   * пробелы по краям ничего не значат;
   * прочерк `'—'` и пустая строка — не число: `None`, а не `0.0`.
2. По списку `RAW`: сумма всего, что разобралось, и сколько значений отброшено. `None` тоже считается отброшенным.

**Формат ответа:** `[сумма, отброшено]` — `float` до двух знаков и целое.

**Подсказка:** `try: ... except ValueError: ...` — ловите именно `ValueError`, а не всё подряд. `None` проверьте до `try`: `float(None)` бросает `TypeError`, а не `ValueError`. `x.split()` делит строку по любым пробелам, включая неразрывный.

In [ ]:
RAW = ["199.00", "1 499,50", None, "—", "349,90", "2\u00a0100", "", "  89.10  "]

# Во втором значении обычный пробел, в шестом — неразрывный (\u00a0).
# Если assert упал — RAW испортился при копировании: верните ячейку из стартового ноутбука.
assert RAW[1][1] == " " and RAW[5][1] == "\u00a0"
def to_value(x: str | None) -> float | None:
    # верните число или None; падать нельзя
    if x is None:
        return None
    cleaned = x.replace(" ", "").replace("\u00a0", "").replace(",", ".")
    try:
        return float(cleaned)
    except ValueError:
        return None
def task7(raw: list) -> list:
    # верните [сумма разобранного, сколько отброшено]
    values = [to_value(x) for x in raw]
    total = sum(value for value in values if value is not None)
    rejected = values.count(None)
    return [float(round(total, 2)), rejected]
answer_7 = task7(RAW)
answer_7

**Вопрос 7.** **Что опаснее.** Представьте ежедневный отчёт о выручке, построенный на этой функции. Что опаснее: функция, которая падает на `'—'`, или функция, которая молча возвращает `None`? Что бы вы добавили в отчёт, чтобы потеря была видна руководителю? Опирайтесь на свой ответ задачи 7: сколько из восьми значений было бы потеряно.

In [ ]:
# Числа для ответа на вопрос 7 считайте здесь — по rows и answer_7.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.

#### ✍️ Ответ на вопрос 7

Опаснее, если функция молча вернет None, потому что если функция упадет, то мы поймем, что есть проблема и будем ее решать. С None код не упадет, но ежедневный отчет будет скорее всего недостоверен и мы примем неверные решения. Для решения я бы добавил сводку по пропускам. 3 из 8 значений было бы утеряно.

### Задача 8. Поправка на день недели

Аналитики из соседней команды прислали коэффициенты дней недели, чтобы сравнивать недели между собой. Прислали только будни — как это обычно и бывает.

**Что посчитать**

1. Для каждого дня недели возьмите сумму `value` по строкам с `ok = True` — как в задаче 3, но **до округления**.
2. Приведите её к `float` и умножьте на коэффициент дня из `WEIGHTS`. Если дня в справочнике нет — коэффициент `1.0`.
3. Сложите все произведения. Округлите до двух знаков **только итог**.

**Формат ответа:** одно число `float` до двух знаков, например `123456.78`.

**Подсказка:** `WEIGHTS.get(day, 1.0)` возвращает коэффициент или `1.0`, если дня нет. `Decimal * float` падает с `TypeError` — сначала `float(...)`.

In [ ]:
WEIGHTS = {"Mon": 1.00, "Tue": 1.05, "Wed": 1.05, "Thu": 1.00, "Fri": 0.95}
def task8(rows: list[dict]) -> float:
    # верните взвешенную сумму
    by_weekday = defaultdict(Decimal)
    for row in rows:
        if row["ok"] and row["value"] is not None:
            day = WEEKDAYS[row["day"].weekday()]
            by_weekday[day] += row["value"]
    weighted_total = sum(
        float(total) * WEIGHTS.get(day, 1.0)
        for day, total in by_weekday.items()
    )
    return round(weighted_total, 2)
answer_8 = task8(rows)
answer_8

**Вопрос 8.** **Что изменила поправка.** Какие дни недели пошли с коэффициентом по умолчанию? На сколько процентов (с точностью до десятых) взвешенная сумма отличается от обычной суммы из задачи 1 — процент считайте от обычной суммы? В какой ситуации на данных ПРАЙМ такой `.get` с умолчанием спрятал бы ошибку, вместо того чтобы о ней сообщить?

In [ ]:
# Числа для ответа на вопрос 8 считайте здесь — по rows и answer_8.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
used_days = {
    WEEKDAYS[row["day"].weekday()]
    for row in rows
    if row["ok"] and row["value"] is not None
}

default_days = [
    day for day in WEEKDAYS
    if day in used_days and day not in WEIGHTS
]
ordinary_total = answer_1[2]
weighted_total = answer_8
print("Дни с коэффициентом по умолчанию:", default_days)
print("Обычная сумма, ₽:", ordinary_total)
print("Взвешенная сумма, ₽:", weighted_total)
if ordinary_total != 0:
    change_percent = (
        (weighted_total - ordinary_total) / ordinary_total * 100
    )
    print("Изменение относительно обычной суммы, %:", round(change_percent, 1))
else:
    print("Процент изменения не определён: обычная сумма равна нулю.")

#### ✍️ Ответ на вопрос 8

Суббота и воскресенье пошли с коэффициентом по умолчанию. Взвешенная сумма больше обычной суммы на 0,5%. Функция .get скроет ошибку в названии ключа, например: Аналитик из соседней команды напишет "Mond": 1.05 вместо "Mon": 1.05.

## Часть 3. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода — служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [ ]:
import json


def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


# Участок из ячейки 1.1 и ответы answer_1 … answer_8; у каждого ответа проверяем формат.
params = {"table_name": TABLE, "segment_column": SEGMENT_COLUMN, "segment": SEGMENT,
          "period_start": PERIOD_START, "period_end": PERIOD_END}
answers = {n: globals().get(f"answer_{n}") for n in range(1, 9)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n}: {answer!r}")
    print("   ✅ формат в порядке" if formats[n] is None else f"   ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» в части 2.")
print("\n--- служебная строка, не удаляйте ---")
print("HW1_ANSWERS=" + json.dumps(
    {"slice": params, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 4. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–3 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: два задания, каждое до 1 балла сверху. Засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом в ячейках ниже и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждое исследование оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, названо, чего ваш расчёт **не** доказывает | 0,3 |
| код исследования выполняется у преподавателя целиком — в том числе на контрольном участке | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Код исследования, как и всё остальное, считайте от значений ячейки 1.1: на контрольном участке он должен отработать без правок.

База умеет считать сама: `count(*)`, `count(distinct …)`, `sum(…)`, условие внутри агрегата — `count(*) filter (where …)`, разбивка — `group by`. Весь сегмент в Python не выгружайте: запрос дольше 2 минут сервер оборвёт. Функция `query` и словарь `COLUMNS` из части 1 здесь пригодятся.

### Исследование А. Можно ли верить вашей тысяче?

Руководитель прочитал сводку и спрашивает: «Вы смотрели тысячу участников, а в сегменте их гораздо больше. Насколько ваши числа верны для всего сегмента? Если в следующем месяце value на участника упадёт на 5 %, мы это заметим по такой тысяче?»

**Что сделать**

1. По **всему** вашему сегменту за окно — одним запросом к базе, не выгружая строки — посчитайте: сколько участников, долю строк с `ok = True`, `value` на участника (сумма `value` по строкам с `ok = True`, делённая на число различных участников).
2. Те же показатели — по вашей тысяче, из `rows`.
3. Возьмите **не меньше 20 других тысяч** участников того же сегмента и окна и посчитайте показатели для каждой.
4. Ответьте: насколько ваша тысяча отличается от всего сегмента? Каков разброс между тысячами? Заметно ли по одной тысяче падение `value` на участника на 5 % — и сколько участников нужно брать, чтобы такое падение было заметно?

**Подсказки.** Другая тысяча — это другое перемешивание. Ячейку 1.4 не меняйте — сделайте копию запроса в своей ячейке: `salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(table=TABLE, **COLUMNS[TABLE])` и вызывайте `query(salted_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END, salt=str(i))` с разными `i`. Разброс удобно описать стандартным отклонением (`statistics.pstdev`). Как разброс зависит от размера выборки — поищите «стандартная ошибка среднего».

In [ ]:
# Исследование А: ваш код. Всё — от значений ячейки 1.1.
from statistics import mean, pstdev
from math import ceil

# 1. Показатели всего сегмента.
# База возвращает только итоговые числа, а не все покупки.

full_sql = """
select
    count(distinct {actor}) as actors_count,
    count(*) as rows_count,
    count(*) filter (where {ok}) as ok_count,
    coalesce(sum({value}) filter (where {ok}), 0) as total_value
from prime.{table}
where {segment_column} = :segment
  and {moment} >= cast(:period_start as date)
  and {moment} < cast(:period_end as date) + 1
"""

full_sql = full_sql.format(table=TABLE, **COLUMNS[TABLE])

full = query(
    full_sql,
    segment=SEGMENT,
    period_start=PERIOD_START,
    period_end=PERIOD_END
)[0]

full_actors = int(full["actors_count"])
full_rows = int(full["rows_count"])
full_ok_share = full["ok_count"] / full_rows
full_value_per_actor = float(full["total_value"] / full_actors)


# 2. Функция для показателей одной выборки.

def sample_metrics(sample: list[dict]) -> dict:
    actors_count = len({row["actor_id"] for row in sample})
    ok_count = sum(1 for row in sample if row["ok"])

    total_value = sum(
        row["value"]
        for row in sample
        if row["ok"] and row["value"] is not None
    )

    return {
        "actors_count": actors_count,
        "ok_share": ok_count / len(sample),
        "value_per_actor": float(total_value / actors_count)
    }


mine_metrics = sample_metrics(rows)

print("ВЕСЬ СЕГМЕНТ")
print("Клиентов:", full_actors)
print("Строк:", full_rows)
print("Доля состоявшихся покупок, %:", round(full_ok_share * 100, 2))
print("Сумма на клиента, ₽:", round(full_value_per_actor, 2))

print("\nМОЯ ВЫБОРКА")
print("Клиентов:", mine_metrics["actors_count"])
print(
    "Доля состоявшихся покупок, %:",
    round(mine_metrics["ok_share"] * 100, 2)
)
print("Сумма на клиента, ₽:", round(mine_metrics["value_per_actor"], 2))

share_difference = (
    mine_metrics["ok_share"] - full_ok_share
) * 100

print(
    "\nРазница долей, процентных пунктов:",
    round(share_difference, 2)
)

if full_value_per_actor != 0:
    value_difference = (
        (mine_metrics["value_per_actor"] - full_value_per_actor)
        / full_value_per_actor * 100
    )
    print("Отклонение суммы на клиента, %:", round(value_difference, 2))


# 3. Двадцать других выборок.
# Добавка salt меняет порядок выбора клиентов.
# Исходный запрос SLICE_SQL при этом не изменяется.

salted_sql = SLICE_SQL.replace(
    "md5(actor_id)",
    "md5(actor_id || :salt)"
).format(table=TABLE, **COLUMNS[TABLE])

samples = []

for i in range(1, 21):
    sample = query(
        salted_sql,
        segment=SEGMENT,
        period_start=PERIOD_START,
        period_end=PERIOD_END,
        salt=f"research_a_{i}"
    )

    metrics = sample_metrics(sample)
    samples.append(metrics)

    print(
        f"Выборка {i}:"
        f" клиентов — {metrics['actors_count']},"
        f" состоявшихся покупок — {metrics['ok_share'] * 100:.2f}%,"
        f" сумма на клиента — {metrics['value_per_actor']:.2f} ₽"
    )


# 4. Разброс показателей между выборками.

shares = [sample["ok_share"] * 100 for sample in samples]
values = [sample["value_per_actor"] for sample in samples]
sizes = {sample["actors_count"] for sample in samples}

print("\nРАЗБРОС МЕЖДУ ВЫБОРКАМИ")
print("Фактические размеры выборок:", sorted(sizes))

print("\nДоля состоявшихся покупок:")
print("Среднее, %:", round(mean(shares), 2))
print("Минимум, %:", round(min(shares), 2))
print("Максимум, %:", round(max(shares), 2))
print("Стандартное отклонение, п.п.:", round(pstdev(shares), 3))

value_std = pstdev(values)

print("\nСумма на клиента:")
print("Среднее, ₽:", round(mean(values), 2))
print("Минимум, ₽:", round(min(values), 2))
print("Максимум, ₽:", round(max(values), 2))
print("Стандартное отклонение, ₽:", round(value_std, 2))


# 5. Приблизительная оценка заметности падения на 5%.
#
# Рабочий ориентир:
# падение должно быть больше двух стандартных отклонений
# показателя между выборками.
#
# Здесь сравниваем выборку с известным показателем всего сегмента.

if full_value_per_actor > 0 and sizes == {1000}:
    drop_5_percent = full_value_per_actor * 0.05
    noise_boundary = 2 * value_std

    print("\nПАДЕНИЕ НА 5%")
    print("Величина падения, ₽ на клиента:", round(drop_5_percent, 2))
    print("Два стандартных отклонения, ₽:", round(noise_boundary, 2))
    print(
        "Падение больше выбранной границы разброса:",
        drop_5_percent > noise_boundary
    )

    if value_std > 0:
        estimated_size = ceil(
            1000 * (noise_boundary / drop_5_percent) ** 2
        )

        print(
            "Приблизительный размер выборки по этому ориентиру:",
            estimated_size
        )

        if estimated_size > full_actors:
            print(
                "Оценка больше всего сегмента."
                " Формула приблизительная и не учитывает,"
                " что при обследовании всего сегмента"
                " случайность отбора исчезает."
            )
    else:
        print(
            "В этих 20 выборках разброс не обнаружен."
            " Этого недостаточно для надёжной оценки нужного размера."
        )
else:
    print(
        "\nОценка размера не выполнена:"
        " нужны положительная сумма на клиента"
        " и выборки ровно по 1000 участников."
    )

#### ✍️ Исследование А

1. Проверил, насколько по нашей выборке можно судить о генеральной совокупности, используя расчет статистических показателей для новых выборок и всего сегмента (покупки через приложение за август 2025 г.) В частности, проверил точность показателей "доли состоявшихся покупок" и "Суммы на клиента". Также проверил, заметно ли по одной тысяче падение value на участника на 5 % — и сколько участников нужно брать, чтобы такое падение было заметно.
2. Расчет включал следующие этапы:
    а) сравнил показатели своей выборки с показателями сегмента;
    б) повторил расчет на 20 других выборках по 1 000 клиентов и оценил разброс показателей (стандартное отклонение);
    в) сравнил величину падения на 5% с двумя стандартными отклонениями показателя между выборками;
    г) оценил необходимый размер выборки, предполагая, что стандартная ошибка уменьшается обратно пропорционально квадратному корню из числа клиентов.
3. Получилось, что разница между моей выборкой и сегментом в доле состоявшихся покупок составляет 97,3-96,72 = 0,58 п.п., отклонение суммы на клиента относительно всего сегмента: -1,85%. Разброс между выборками: стандартное отклонение по доле состоявшихся покупок - 0,461 п.п., по сумме на клиента - 70,13 руб. Величина падения при 5% - 102,91 руб./клиент. Приблизительный размер выборки - 1 858 клиентов.
4. По моей выборке сумма на клиента ниже показателя всего сегмента на 1,85%, а доля состоявшихся покупок выше на 0,58 п.п., но новые выборки показывают, что результаты зависят от состава выборки.
Падение на 5% меньше двух стандартных отклонений, поэтому необходимо увеличить выборку приблизительно до 1 858 клиентов.
Важно отметить, что расчет не доказывает, что выборка в 1 858 клиентов гарантирует обнаруение падения. Рассчитанное значение относится только к данным за август, если поведение покупателей поменяется, то значение будет необходимо пересчитать.

### Исследование Б. Ваш сегмент особенный?

Маркетинг заметил, что в одних сегментах участники приносят заметно больше, чем в других, и хочет перераспределить бюджет в пользу «ценных» сегментов. Руководитель просит проверить: правда ли ваш сегмент отличается от соседних — и чем именно.

**Что сделать**

1. Для **всех сегментов** вашей таблицы за ваше окно посчитайте: число участников, строк на участника, долю строк с `ok = True`, `value` на одну строку с `ok = True` и `value` на участника.
2. Разложите `value` на участника на множители и найдите, какой из них даёт различие между сегментами.
3. Проверьте, держится ли картина в предыдущем окне той же длины. Данные начинаются 6 января 2025 года: если предыдущее окно начинается раньше, оно неполное — тогда сравнивайте доли и значения на участника или на строку, а не число участников и строк.
4. Вывод для маркетинга: «ценный» ли ваш сегмент; что на самом деле различается; какое решение по бюджету из этого следует — и чего ваши данные не доказывают.

**Подсказка.** Здесь нужны не выгрузка `rows`, а агрегаты по всей таблице за окно с разбивкой `group by` колонке сегмента.

In [ ]:
# Исследование Б: ваш код. Всё — от значений ячейки 1.1.
from datetime import timedelta


# 1. Один запрос для показателей всех сегментов за заданное окно.

segments_sql = """
select
    {segment_column} as segment,
    count(distinct {actor}) as actors_count,
    count(*) as rows_count,
    count(*) filter (where {ok}) as ok_count,
    coalesce(sum({value}) filter (where {ok}), 0) as total_value
from prime.{table}
where {moment} >= cast(:period_start as date)
  and {moment} < cast(:period_end as date) + 1
group by {segment_column}
order by {segment_column}
"""

segments_sql = segments_sql.format(
    table=TABLE,
    **COLUMNS[TABLE]
)


def segment_metrics(period_start: date, period_end: date) -> dict:
    result = query(
        segments_sql,
        period_start=period_start.isoformat(),
        period_end=period_end.isoformat()
    )

    metrics = {}

    for row in result:
        actors = int(row["actors_count"])
        rows_count = int(row["rows_count"])
        ok_count = int(row["ok_count"])
        total_value = row["total_value"]

        metrics[row["segment"]] = {
            "actors": actors,
            "rows_per_actor": rows_count / actors if actors else None,
            "ok_share": ok_count / rows_count if rows_count else None,
            "value_per_ok": (
                float(total_value / ok_count) if ok_count else None
            ),
            "value_per_actor": (
                float(total_value / actors) if actors else None
            )
        }

    return metrics


# 2. Текущее и предыдущее окна одинаковой длины.

current_start = date.fromisoformat(PERIOD_START)
current_end = date.fromisoformat(PERIOD_END)

period_days = (current_end - current_start).days + 1

previous_end = current_start - timedelta(days=1)
previous_start = current_start - timedelta(days=period_days)

current_metrics = segment_metrics(current_start, current_end)
previous_metrics = segment_metrics(previous_start, previous_end)

print("Текущее окно:", current_start, "—", current_end)
print("Предыдущее окно:", previous_start, "—", previous_end)
print("Длительность каждого окна:", period_days, "дней")

# Дата начала базы прямо указана в условии исследования.
data_start = date(2025, 1, 6)

if previous_start < data_start:
    print(
        "Внимание: предыдущее окно начинается раньше данных базы."
        " Сравнивать абсолютные количества напрямую нельзя."
        " Доли и показатели на клиента тоже требуют осторожности:"
        " фактическая длительность наблюдения отличается."
    )


# 3. Вывод пяти показателей по каждому сегменту.

def show_metrics(metrics: dict, title: str) -> None:
    print("\n" + title)

    if not metrics:
        print("За это окно данных нет.")
        return

    for segment, values in metrics.items():
        print("\nСегмент:", segment)
        print("Участников:", values["actors"])

        for key, label in [
            ("rows_per_actor", "Строк на участника"),
            ("ok_share", "Доля состоявшихся событий"),
            ("value_per_ok", "Value на состоявшееся событие"),
            ("value_per_actor", "Value на участника")
        ]:
            value = values[key]

            if value is None:
                print(label + ": не определено")
            elif key == "ok_share":
                print(label + ", %:", round(value * 100, 2))
            else:
                print(label + ":", round(value, 2))

        # Проверяем разложение на три множителя.
        factors = [
            values["rows_per_actor"],
            values["ok_share"],
            values["value_per_ok"]
        ]

        if all(value is not None for value in factors):
            product = factors[0] * factors[1] * factors[2]
            print("Произведение трёх множителей:", round(product, 2))


show_metrics(current_metrics, "ТЕКУЩЕЕ ОКНО")
show_metrics(previous_metrics, "ПРЕДЫДУЩЕЕ ОКНО")


# 4. Сравнение твоего сегмента с каждым другим.
# Показываем отношения показателей:
# больше 1 — показатель твоего сегмента выше;
# меньше 1 — ниже.

def compare_segments(metrics: dict, title: str) -> None:
    print("\n" + title)

    if SEGMENT not in metrics:
        print("Твоего сегмента нет в этом окне.")
        return

    mine = metrics[SEGMENT]

    for segment, other in metrics.items():
        if segment == SEGMENT:
            continue

        print(f"\nОтношения показателей: {SEGMENT} / {segment}")

        for key, label in [
            ("rows_per_actor", "Строк на участника"),
            ("ok_share", "Доля состоявшихся событий"),
            ("value_per_ok", "Value на состоявшееся событие"),
            ("value_per_actor", "Value на участника")
        ]:
            mine_value = mine[key]
            other_value = other[key]

            if (
                mine_value is None
                or other_value is None
                or other_value == 0
            ):
                print(label + ": отношение не определено")
            else:
                ratio = mine_value / other_value
                print(label + ":", round(ratio, 3))


compare_segments(
    current_metrics,
    "СРАВНЕНИЕ СЕГМЕНТОВ В ТЕКУЩЕМ ОКНЕ"
)

compare_segments(
    previous_metrics,
    "СРАВНЕНИЕ СЕГМЕНТОВ В ПРЕДЫДУЩЕМ ОКНЕ"
)

In [ ]:
import pandas as pd
from IPython.display import display

table_rows = []

for period, metrics in [
    (f"{previous_start} — {previous_end}", previous_metrics),
    (f"{current_start} — {current_end}", current_metrics)
]:
    for segment, values in metrics.items():
        table_rows.append({
            "Период": period,
            "Сегмент": segment,
            "Клиентов": values["actors"],
            "Покупок на клиента": values["rows_per_actor"],
            "Состоявшихся, %": (
                values["ok_share"] * 100
                if values["ok_share"] is not None else None
            ),
            "Средняя сумма состоявшейся покупки, ₽": values["value_per_ok"],
            "Сумма на клиента, ₽": values["value_per_actor"]
        })

comparison_table = pd.DataFrame(table_rows)

styled_table = (
    comparison_table.style
    .format(
        {
            "Клиентов": "{:,.0f}",
            "Покупок на клиента": "{:.2f}",
            "Состоявшихся, %": "{:.2f}",
            "Средняя сумма состоявшейся покупки, ₽": "{:,.2f}",
            "Сумма на клиента, ₽": "{:,.2f}"
        },
        thousands=" ",
        decimal=",",
        na_rep="—"
    )
    .hide(axis="index")
    .set_caption("Сравнение каналов покупок за два периода")
    .set_properties(**{
        "padding": "10px",
        "text-align": "right",
        "border-bottom": "1px solid #dddddd"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#234E70"),
                ("color", "white"),
                ("padding", "10px"),
                ("text-align", "center")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("font-size", "18px"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("padding", "10px 0")
            ]
        }
    ])
)

display(styled_table)

##### ✍️ Исследование Б

1. Я проверил ценность сегмента (приложения) по сравнению с другими сегментами по сумме состоявшихся покупок на клиента и факторы, образующие данный показатель. Далее я проверил, подтверждаются ли мои выводы по различиям показателей по сегментам в разрезе месяца (июль), предшествующего моему (августу)
2. Рассчитал количество клиентов, количество строк на участника, долю состоявшихся покупок, Value на состоявшуюся покупку, Value на клиента для каждого сегмента в своем окне и предыдущем.Также провел разложение суммы клиента на множители (строки на клиента * доля состоявшихся покупок * средняя сумма состоявшейся покупки). Далее сравнил данные показатели сегментов относительно друг друга.
3. Получилось, что количество клиентов больше, покупок на клиента в моем сегменте больше, чем в остальных на 22,5 - 34,8%. Средняя сумма состоявшихся покупки больше, чем в остальных сегментах на 0,08-0,21%, сумма покупок на клиента больше на 22,5-34,9%. Доля состояшихся покупок немного ниже остальных сегментов и отличается на 0,07-0,19 п.п. Такая тенденция соблюдается и в предшествующем окне,за исключением средней суммы состоявшейся покупки, в данном случае в сегменте app этот показатель меньше на 0,7-1,5%. 
4. Анализ показал, что клиентов, которые покупают через приложение больше и они совершают больше количество покупок. Основное преимущество сегмента - количество строк на клиента, именно этот показатель дает большую сумму на клиента, что делает сегмент ценным. Обосновано закладывать больше бюджета под сегмент "app".
Важно отметить, что расчет не доказывает, что увеличение бюджета для сегмента - увеличит прибыль компании. Также необходимо учитывать, что нельзя считать сегменты независимыми группами разных людей и складывать их численности как число уникальных клиентов. 

## Напоследок

Что вас удивило в вашем участке? Одна-две фразы в ячейке ниже — по желанию и без баллов.

И пройдите, пожалуйста, **[короткий опрос о том, как вам это задание](https://forms.gle/5tNT4P4diHVyzhdFA)** — пара минут. Следующие задания соберу с учётом ответов.

#### ✍️ Что удивило

Удивило, что я мог использовать все эти методики у себя на работе